In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

# 01 · Data Exploration — KAIROS synthetic multi-source data

KAIROS ships a synthetic data generator (`backend/demo/synthetic.py`) that produces **hourly business metrics
from four simulated sources** (marketing CSV, ERP database, logistics REST API, customer-service Google Sheet)
with a **known causal structure** (`TRUE_EDGES`) and **injected incidents**. Because the ground truth is known,
the same data is used to evaluate anomaly detection (notebook 02) and causal discovery (notebook 03).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from backend.demo.synthetic import generate, TRUE_EDGES, METRICS

pd.set_option('display.width', 140)

## 1. Generate the data
`generate()` returns a long-format frame `(source, metric_name, timestamp, value)`: 336 hourly points (14 days) per
metric by default. A fixed `end` timestamp and `seed` make the notebook reproducible.

In [ ]:
END = pd.Timestamp('2026-10-01 00:00', tz='UTC')
df = generate(periods=336, seed=42, end=END, incidents=True)
print(df.shape)
df.head()

In [ ]:
print('time range:', df['timestamp'].min(), '->', df['timestamp'].max())
print('metrics:', df['metric_name'].nunique(), '| sources:', df['source'].nunique())
df.dtypes

## 2. Metrics per source

In [ ]:
per_source = (df.groupby('source')['metric_name']
                .agg(n_metrics='nunique', metrics=lambda s: ', '.join(sorted(s.unique()))))
per_source

## 3. Descriptive statistics per metric
Including a robust spread (MAD) — the same robust scale used by the anomaly detector.

In [ ]:
def mad(x):
    return 1.4826 * np.median(np.abs(x - np.median(x)))

stats = (df.groupby(['source', 'metric_name'])['value']
           .agg(count='count', mean='mean', std='std', median='median', mad=mad, min='min', max='max')
           .round(3))
stats

Data quality check — every metric should have one value per hour with no gaps or duplicates.

In [ ]:
quality = df.groupby('metric_name').agg(
    n=('value', 'size'),
    nulls=('value', lambda s: s.isna().sum()),
    dup_ts=('timestamp', lambda s: s.duplicated().sum()),
)
assert (quality['nulls'] == 0).all() and (quality['dup_ts'] == 0).all()
quality.T

## 4. Plot a few series
The marketing chain (`ad_spend_usd → click_through_rate → conversion_rate → order_count → daily_revenue`) and the
operations chain (`delivery_delay_hrs → support_tickets_open → ...`). Incident A (ad-spend collapse, 14 h before the end)
and incident B (delivery-delay spike, 8 h before the end) should be visible propagating downstream.

In [ ]:
wide = df.pivot_table(index='timestamp', columns='metric_name', values='value').sort_index()

chains = {
    'Marketing chain': ['ad_spend_usd', 'click_through_rate', 'conversion_rate', 'order_count', 'daily_revenue'],
    'Operations chain': ['delivery_delay_hrs', 'support_tickets_open', 'avg_response_time_hrs', 'nps_score',
                         'customer_churn_rate'],
}
inc_a = wide.index[-14]
inc_b = wide.index[-8]
for title, cols in chains.items():
    fig, axes = plt.subplots(len(cols), 1, figsize=(12, 1.9 * len(cols)), sharex=True)
    for ax, c in zip(axes, cols):
        ax.plot(wide.index, wide[c], lw=1)
        ax.axvline(inc_a, color='tab:red', ls='--', lw=0.8)
        ax.axvline(inc_b, color='tab:orange', ls='--', lw=0.8)
        ax.set_ylabel(c, rotation=0, ha='right', fontsize=8)
    axes[0].set_title(f'{title} (red: incident A, orange: incident B)')
    plt.tight_layout()
    plt.show()

Zoom into the last 48 hours, standardised so the metrics share one axis.

In [ ]:
last = wide.iloc[-48:]
z = (last - wide.mean()) / wide.std()
fig, ax = plt.subplots(figsize=(12, 4))
for c in chains['Marketing chain']:
    ax.plot(z.index, z[c], label=c)
ax.axvline(inc_a, color='k', ls='--', lw=0.8)
ax.set_ylabel('z-score')
ax.set_title('Incident A propagating with a ~1 h lag per hop')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 5. Daily seasonality
Some metrics have an injected hour-of-day cycle; the anomaly detector models this with an hour-of-day profile.

In [ ]:
hourly = wide.groupby(wide.index.hour).mean()
hourly_z = (hourly - hourly.mean()) / wide.std()
hourly_z[['ad_spend_usd', 'order_count', 'daily_revenue', 'support_tickets_open', 'gross_margin_pct']].plot(
    figsize=(10, 3.5), title='Mean value by hour of day (standardised)')
plt.xlabel('hour (UTC)')
plt.tight_layout()
plt.show()

## 6. Correlation is not causation
The correlation matrix is symmetric: it cannot tell which metric drives which. That is why KAIROS uses temporal causal discovery (notebook 03).

In [ ]:
corr = wide.corr()
fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr)))
ax.set_xticklabels(corr.columns, rotation=90, fontsize=7)
ax.set_yticks(range(len(corr)))
ax.set_yticklabels(corr.columns, fontsize=7)
fig.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('Pearson correlation (symmetric — no direction)')
plt.tight_layout()
plt.show()

## 7. Ground-truth causal edges
`TRUE_EDGES` lists `(cause, effect, coefficient on the standardised scale, lag in hours)`. `gross_margin_pct` is independent.

In [ ]:
truth = pd.DataFrame(TRUE_EDGES, columns=['cause', 'effect', 'coefficient', 'lag_h'])
truth['cause_source'] = truth['cause'].map(lambda m: METRICS[m][0])
truth['effect_source'] = truth['effect'].map(lambda m: METRICS[m][0])
truth['cross_source'] = truth['cause_source'] != truth['effect_source']
truth

In [ ]:
print(f"{len(truth)} true edges, {truth['cross_source'].sum()} of them cross source boundaries, "
      f"{(truth['coefficient'] < 0).sum()} negative")

**Takeaway:** some causal edges cross system boundaries (conversion_rate in the marketing CSV → order_count in the ERP; delivery_delay_hrs in the logistics API → support_tickets_open in the CS sheet), links that no single-source BI dashboard would connect.